In [1]:
# Measure the time it takes every cell to run
%pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.2 MB/s eta 0:00:00
time: 302 µs (started: 2026-03-26 13:18:31 +00:00)


In [3]:
# first figure out how to convert CT file into a .nrrd file
%pip install SimpleITK
import SimpleITK as sitk
import os

def convert_CT_to_nrrd(input_folder, output_folder):
  # assume input_folder and output_folder exist; case handling elsewhere
  # assume input_folder is the folder of a CT series
  reader = sitk.ImageSeriesReader()
  dicom_names = reader.GetGDCMSeriesFileNames(input_folder)
  reader.SetFileNames(dicom_names)
  ct_image = reader.Execute()
  sitk.WriteImage(ct_image, os.path.join(output_folder, 'CT.nrrd'))

%pip install dcmqi
import subprocess
import json
import glob

def convert_SEG_to_nrrd(input_file, output_folder, prefix="SEG"):
  command = ["seg2itkimage", "--inputDICOM", input_file, "--outputDirectory", output_folder, "--outputType", "nrrd", "--prefix", prefix]
  try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    return True
  except suprocess.CalledProcessError as e:
    print(f"Conversion of SEG to .nrrd failed: {e}")
    return False

def identify_and_rename_masks(output_folder, prefix="SEG"):
  # Parse JSON to identify Lung and Nodule segments and rename .nrrd files accordingly
  json_files = glob.glob(os.path.join(output_folder, f"{prefix}-*.json"))
  masks = []
  for json_file in json_files:
    with open(json_file, 'r') as f:
      metadata = json.load(f)
      segments = metadata.get('segments', [])
      for segment in segments:
        segment_label = segment.get('SegmentLabel', '').lower()
        old_name = json_file.replace(".json", ".nrrd")
        if "lung" in segment_label:
          new_name = os.path.join(output_folder, "lung_mask.nrrd")
          os.rename(old_name, new_name)
          masks.append(new_name)
        elif "nodule" in segment_label:
          new_name = os.path.join(output_folder, "nodule_mask.nrrd")
          os.rename(old_name, new_name)
          masks.append(new_name)
    os.remove(json_file)
  return masks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 MB 11.3 MB/s eta 0:00:00
time: 14.9 s (started: 2026-03-26 13:18:54 +00:00)


In [ ]:
# Convert CT and SEG series to .nrrd files for each patient and timestep
base_path = '/content/malignant_trial/nlst'

for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    if not os.path.isdir(patient_path):
        continue

    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        if not os.path.isdir(timestep_path):
            continue

        ct_series_path = None
        seg_series_path = None

        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path

        if ct_series_path:
            try:
                convert_CT_to_nrrd(ct_series_path, timestep_path)
                print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {timestep_path}")
            except Exception as e:
                print(f"Failed to convert CT for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"CT series folder not found for patient {patient_id}, timestep {timestep_dir}. Skipping CT and SEG conversion for this timestep.")
            continue # Skip to next timestep if CT is not found

        if seg_series_path:
            try:
                seg_dcm_files = [f for f in os.listdir(seg_series_path) if f.endswith('.dcm')] # should be only one .dcm always, so remove this?
                if not seg_dcm_files:
                    print(f"No DICOM SEG files found in SEG series for patient {patient_id}, timestep {timestep_dir}")
                    continue
                seg_file = os.path.join(seg_series_path, seg_dcm_files[0])
                if convert_SEG_to_nrrd(seg_file, timestep_path):
                  masks = identify_and_rename_masks(timestep_path)
                  print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
            except Exception as e:
                print(f"Failed to convert SEG for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"SEG series folder not found for patient {patient_id}, timestep {timestep_dir}")